# S6 · 2025 holdout and experiment-triage boundary

The proposed “Narrative Attention Engine” is not validated for deployment. This stage compares narrative features against a controls-only baseline out of time and retains scores only as an interpretable experiment-triage prototype.

In [1]:
from pathlib import Path
import json, time
import pandas as pd
import numpy as np
import psutil
from IPython.display import display, Image

ROOT = Path.cwd().parent
OUTPUTS = ROOT / "outputs"
AUDIT = ROOT / "audit"
LOGS = ROOT / "logs"
LOGS.mkdir(exist_ok=True)
RUN_LOG = LOGS / "run_log.txt"

def checkpoint(label, *frames, started=None):
    elapsed = time.time() - started if started is not None else 0.0
    shapes = [getattr(x, "shape", None) for x in frames]
    nulls = []
    for frame in frames:
        if hasattr(frame, "isna"):
            nulls.append(round(float(frame.isna().mean(numeric_only=False).mean()), 6))
    rss = psutil.Process().memory_info().rss / 1024**3
    line = f"{label} | shapes={shapes} | mean_null_rates={nulls} | rss_gib={rss:.3f} | elapsed_s={elapsed:.3f}"
    print(line)
    with RUN_LOG.open("a", encoding="utf-8") as stream:
        stream.write(line + "\n")

t0 = time.time()
print(f"Audit root: {ROOT}")
checkpoint("setup", started=t0)

Audit root: <PROJECT_ROOT>/5_最终交付包/_rebuild/MA-Hackathon-Final-2026-09-04
setup | shapes=[] | mean_null_rates=[] | rss_gib=0.113 | elapsed_s=0.000


In [2]:
t0 = time.time()
holdout = pd.read_csv(OUTPUTS / "holdout_validation.csv")
calibration = pd.read_csv(OUTPUTS / "holdout_calibration.csv")
actions = pd.read_csv(OUTPUTS / "engine_action_summary.csv")
fairness = pd.read_csv(OUTPUTS / "fairness_diagnostics.csv")
display(holdout)
display(actions)
display(fairness)
checkpoint("S6 holdout diagnostics", holdout, calibration, actions, fairness, started=t0)

,task,evaluation_slice,model,metric,value,n
0,funding_duration,2025_all_observed,controls_only,mae_log_hours,0.964600,133409
1,funding_duration,2025_all_observed,controls_only,median_ae_log_hours,0.811130,133409
2,funding_duration,2025_all_observed,controls_only,rmse_log_hours,1.225251,133409
3,funding_duration,2025_all_observed,controls_only,r2_log_hours,0.552387,133409
4,funding_duration,2025_all_observed,controls_only,mae_hours,161.986893,133409
5,funding_duration,2025_all_observed,controls_only,median_ae_hours,44.479214,133409
6,funding_duration,2025_all_observed,controls_plus_narrative,mae_log_hours,0.978036,133409
7,funding_duration,2025_all_observed,controls_plus_narrative,median_ae_log_hours,0.813352,133409
8,funding_duration,2025_all_observed,controls_plus_narrative,rmse_log_hours,1.246147,133409
9,funding_duration,2025_all_observed,controls_plus_narrative,r2_log_hours,0.536989,133409


,triage_action,loans,mean_predicted_72h,loan_pct
0,monitor; no automated intervention,51146,0.581204,38.337743
1,test listing cadence or diversify related-loan...,34363,0.445918,25.757633
2,test platform-side narrative refresh,21621,0.387084,16.206553
3,test exposure support for a distinctive but sl...,14050,0.138027,10.531523
4,offer recurring-language template support; do ...,12229,0.514983,9.166548


,dimension,group,loans,observed_72h_rate,predicted_72h_rate,calibration_gap,median_absolute_error_hours,use_boundary
0,gender,female,115334,0.538644,0.485144,-0.053500,42.738550,diagnostic guardrail only; no protected-group ...
1,gender,male,17664,0.364300,0.301174,-0.063126,106.206774,diagnostic guardrail only; no protected-group ...
2,sector,Agriculture,46312,0.503412,0.444421,-0.058990,54.209452,diagnostic guardrail only; no protected-group ...
3,sector,Arts,1111,0.539154,0.568918,0.029764,32.894462,diagnostic guardrail only; no protected-group ...
4,sector,Clean Energy,2116,0.716446,0.737909,0.021463,15.406140,diagnostic guardrail only; no protected-group ...
5,sector,Clothing,3862,0.386069,0.373985,-0.012085,74.727420,diagnostic guardrail only; no protected-group ...
6,sector,Education,1530,0.475163,0.487766,0.012603,79.599675,diagnostic guardrail only; no protected-group ...
7,sector,Food,28209,0.484207,0.452660,-0.031547,50.039930,diagnostic guardrail only; no protected-group ...
8,sector,Health,1419,0.395349,0.308658,-0.086691,102.389363,diagnostic guardrail only; no protected-group ...
9,sector,Housing,4202,0.683008,0.696492,0.013484,11.296725,diagnostic guardrail only; no protected-group ...


S6 holdout diagnostics | shapes=[(44, 6), (20, 6), (5, 4), (31, 8)] | mean_null_rates=[0.0, 0.0, 0.0, 0.0] | rss_gib=0.114 | elapsed_s=0.011


In [3]:
t0 = time.time()
def metric(task, slice_name, model, name):
    row = holdout[(holdout.task == task) & (holdout.evaluation_slice == slice_name) & (holdout.model == model) & (holdout.metric == name)]
    return float(row.value.iloc[0])

assert metric("funding_duration", "2025_at_least_35d_followup", "controls_plus_narrative", "mae_log_hours") > metric("funding_duration", "2025_at_least_35d_followup", "controls_only", "mae_log_hours")
assert metric("fast_funding_72h", "2025_72h_eligible", "controls_plus_narrative", "brier") > metric("fast_funding_72h", "2025_72h_eligible", "controls_only", "brier")
assert metric("fast_funding_72h", "2025_72h_eligible", "controls_plus_narrative", "roc_auc") < metric("fast_funding_72h", "2025_72h_eligible", "controls_only", "roc_auc")
print("NO-DEPLOYMENT: narrative features do not outperform controls on log-MAE, Brier, or AUC.")
print("Permitted use: generate prospective platform experiments; forbidden use: borrower ranking, approval, or penalty.")
checkpoint("S6 deployment boundary", holdout, started=t0)

NO-DEPLOYMENT: narrative features do not outperform controls on log-MAE, Brier, or AUC.
Permitted use: generate prospective platform experiments; forbidden use: borrower ranking, approval, or penalty.
S6 deployment boundary | shapes=[(44, 6)] | mean_null_rates=[0.0] | rss_gib=0.114 | elapsed_s=0.002
